# Sensor Upload Workflow

This notebook rebuilds the SHM sensor upload flow as an explicit, auditable dry-run that stays on the public `owi.metadatabase.shm` surface instead of using ad hoc demo helpers.

**What this notebook covers**

- **Import** the SHM API, typed services, registry, serializers, and upload helpers.
- **Configure** the packaged Norther sensor data paths, token source, target turbines, and dry-run API root.
- **Construct** a `ShmAPI`-compatible dry-run transport so the rest of the workflow exercises the same method names as a live backend.
- **Preview** how raw packaged JSON becomes typed SHM payloads through the registry and serializers.
- **Upload** sensor types, sensors, and sensor calibrations through `ShmSensorUploader`.
- **Retrieve** the persisted dry-run rows back through `ApiShmRepository` and `SensorService`.

**How to use it**

- Run the notebook top to bottom.
- Update the constants cell when you want to point to a different workspace or packaged dataset.
- Use the final typed retrieval cells to verify exactly what would be sent to the backend in a live run.


## Mermaid Color Legend

The workflow diagrams in this notebook use a consistent visual language.

- <span style="color:#2B6F77;"><strong>Blue nodes</strong></span>: API calls, services, or repository interactions.
- <span style="color:#5E8C61;"><strong>Green nodes</strong></span>: validated outputs, typed records, or persisted dry-run rows.
- <span style="color:#C08B3E;"><strong>Amber nodes</strong></span>: transformations, serializers, or intermediate payload preparation.
- <span style="color:#C47A2C;"><strong>Orange decision/request nodes</strong></span>: configuration choices or upload stages.
- <span style="color:#365A73;"><strong>Blue-grey connectors</strong></span>: data flow between notebook stages.

*Read the diagrams from top to bottom to follow the upload lifecycle from packaged JSON to typed round-trip inspection.*


## Imports

These imports assemble the SHM surfaces that the dry-run workflow should prove out.

**This step prepares**

- **Environment setup** with `Path`, `pandas`, and notebook display helpers.
- **Shared token loading** through `load_token_from_env_file` from the core SDK utilities.
- **Transport and upload orchestration** through `ShmAPI` and `ShmSensorUploader`.
- **Typed retrieval and validation** through `ApiShmRepository`, `SensorService`, `default_registry`, and `DEFAULT_SERIALIZERS`.

*Outcome:* later cells can focus on the sensor workflow itself instead of repeating setup code.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from owi.metadatabase._utils.utils import load_token_from_env_file

from owi.metadatabase.shm import (
    DEFAULT_SERIALIZERS,
    ApiShmRepository,
    SensorService,
    ShmAPI,
    ShmEntityName,
    ShmEntityService,
    ShmQuery,
    ShmSensorUploader,
    default_registry,
    load_json_data,
)


## Constants

This section defines the runtime configuration for the dry-run workflow.

**Review these values before execution**

- **`WORKSPACE_ROOT`**: repository root resolved from the `scripts/` folder.
- **`DEFAULT_ENV_FILE`** and **`TOKEN`**: live-style authentication inputs shared across notebooks.
- **`BASE_URL`**: dry-run API root used when constructing the `ShmAPI` subclass.
- **`TARGET_TURBINES`** and **`PERMISSION_GROUP_IDS`**: the subset of packaged Norther data exercised by the notebook.

*Outcome:* the notebook has one explicit configuration block that mirrors the Results notebook structure while still staying dry-run safe.*


In [11]:
WORKSPACE_ROOT = Path.cwd().resolve().parent
DEFAULT_ENV_FILE = WORKSPACE_ROOT / ".env"
TOKEN = load_token_from_env_file(DEFAULT_ENV_FILE)
BASE_URL = "https://owimetadatabase-dev.azurewebsites.net/api/v1"
SENSORS_ROOT = WORKSPACE_ROOT / "scripts" / "data" / "Norther" / "sensors"
SIGNAL_SENSOR_MAP_PATH = WORKSPACE_ROOT / "scripts" / "data" / "Norther" / "signal_sensor_map.json"
TARGET_TURBINES = ["NRTC01", "NRTC03"]
PERMISSION_GROUP_IDS = [1, 2]

config_summary = pd.DataFrame(
    [
        {
            "workspace_root": str(WORKSPACE_ROOT),
            "sensors_root": str(SENSORS_ROOT),
            "signal_sensor_map": str(SIGNAL_SENSOR_MAP_PATH),
            "api_root": BASE_URL,
            "token_available": TOKEN is not None,
            "target_turbines": ", ".join(TARGET_TURBINES),
            "permission_groups": PERMISSION_GROUP_IDS,
        }
    ]
).T.reset_index().rename(columns={"index": "setting", 0: "value"}).set_index("setting")

display(config_summary)


,value
setting,
workspace_root,/home/pietro.dantuono@24SEA.local/Projects/SMA...
sensors_root,/home/pietro.dantuono@24SEA.local/Projects/SMA...
signal_sensor_map,/home/pietro.dantuono@24SEA.local/Projects/SMA...
api_root,https://owimetadatabase-dev.azurewebsites.net/...
token_available,True
target_turbines,"NRTC01, NRTC03"
permission_groups,"[1, 2]"


## Dry-Run Transport and Typed Service Stack

The goal is to keep the upload flow realistic without calling a live backend.

**What happens here**

- Subclass `ShmAPI` with an in-memory persistence layer.
- Keep the public `create_*`, `get_*`, and `list_*` method signatures intact.
- Route retrieval through `ApiShmRepository` and `SensorService` exactly as a typed client would.
- Use the same registry and serializers that a live repository-backed workflow would use.

*Why this matters:* the notebook demonstrates the real SHM package architecture instead of a parallel demo-only abstraction.


```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["Constants<br/>BASE_URL, TOKEN, packaged paths"] --> B["DryRunSensorApi<br/>subclasses ShmAPI"]
    B --> C["ApiShmRepository"]
    C --> D["ShmEntityService"]
    D --> E["SensorService"]
    B --> F["ShmSensorUploader"]
    G["default_registry"] --> D
    H["DEFAULT_SERIALIZERS"] --> D
    F --> B
    E --> I["Typed SensorRecord / SensorTypeRecord / SensorCalibrationRecord"]

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class B,C,D,E,F api;
    class A,G,H transform;
    class I output;
```


In [12]:
class DryRunSensorApi(ShmAPI):
    """In-memory sensor-domain transport that preserves the public ShmAPI surface."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._next_id = 1
        self._sensor_types: list[dict[str, object]] = []
        self._sensors: list[dict[str, object]] = []
        self._sensor_calibrations: list[dict[str, object]] = []

    def _allocate_id(self) -> int:
        value = self._next_id
        self._next_id += 1
        return value

    def _result(self, rows: list[dict[str, object]], object_id: int | None = None) -> dict[str, object]:
        frame = pd.DataFrame(rows)
        result_id = object_id
        if result_id is None and not frame.empty and "id" in frame.columns:
            result_id = int(frame.iloc[0]["id"])
        return {
            "data": frame,
            "exists": not frame.empty,
            "id": result_id,
            "response": None,
        }

    @staticmethod
    def _normalize_sensor_row(row: dict[str, object] | dict[str, str | int | None]) -> dict[str, object]:
        normalized = dict(row)
        if normalized.get("serial_number") is not None:
            normalized["serial_number"] = str(normalized["serial_number"])
        if normalized.get("cabinet") is not None:
            normalized["cabinet"] = str(normalized["cabinet"])
        return normalized

    def _filter(self, rows: list[dict[str, object]], filters: dict[str, object]) -> list[dict[str, object]]:
        normalized_filters = self._normalize_sensor_row(filters)
        return [
            row
            for row in rows
            if all(row.get(key) == value for key, value in normalized_filters.items())
        ]

    def create_sensor_type(self, payload, files=None):
        row = dict(payload)
        row["id"] = self._allocate_id()
        if files:
            row["file"] = next(iter(files.values()))[0]
            for _, (_, handle) in files.items():
                handle.close()
        self._sensor_types.append(row)
        return self._result([row])

    def list_sensor_types(self, **kwargs):
        return self._result(self._filter(self._sensor_types, kwargs))

    def get_sensor_type(self, **kwargs):
        rows = self._filter(self._sensor_types, kwargs)
        return self._result(rows[:1], int(rows[0]["id"]) if rows else None)

    def create_sensor(self, payload):
        row = self._normalize_sensor_row(dict(payload))
        row["id"] = self._allocate_id()
        self._sensors.append(row)
        return self._result([row])

    def list_sensors(self, **kwargs):
        return self._result(self._filter(self._sensors, kwargs))

    def get_sensor(self, **kwargs):
        rows = self._filter(self._sensors, kwargs)
        return self._result(rows[:1], int(rows[0]["id"]) if rows else None)

    def create_sensor_calibration(self, payload, files=None):
        row = dict(payload)
        row["id"] = self._allocate_id()
        if files:
            row["file"] = next(iter(files.values()))[0]
            for _, (_, handle) in files.items():
                handle.close()
        self._sensor_calibrations.append(row)
        return self._result([row])

    def list_sensor_calibrations(self, **kwargs):
        return self._result(self._filter(self._sensor_calibrations, kwargs))

    def get_sensor_calibration(self, **kwargs):
        rows = self._filter(self._sensor_calibrations, kwargs)
        return self._result(rows[:1], int(rows[0]["id"]) if rows else None)


## Packaged Data Import and Payload Preview

This stage loads the Norther sample files and shows how the typed SHM layer sees them before upload.

**What happens here**

- Load the packaged sensor type, sensor inventory, signal-to-sensor, and calibration files.
- Inspect the sensor-domain registry entries that control endpoint and model selection.
- Serialize the first sensor-type mapping through `DEFAULT_SERIALIZERS` to preview the upload payload shape.
- Summarize the packaged dataset that the dry-run uploader will consume.

*Outcome:* the notebook moves from raw packaged JSON to explicit, typed payload preparation.


```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["sensor_types.json"] --> D["load_json_data"]
    B["sensors.json"] --> D
    C["sensor_calibrations.json + signal_sensor_map.json"] --> D
    D --> E["Packaged Python mappings"]
    F["default_registry"] --> G["sensor entity definitions"]
    H["DEFAULT_SERIALIZERS[sensor_type]"] --> I["preview upload payload"]
    E --> I
    E --> J["dataset summary tables"]

    classDef source fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class A,B,C,D,F,H source;
    class E,G,I transform;
    class J output;
```


In [13]:
sensor_types_data = load_json_data(SENSORS_ROOT / "sensor_types.json")
sensors_data = load_json_data(SENSORS_ROOT / "sensors.json")
sensor_calibration_map = load_json_data(SENSORS_ROOT / "sensor_calibrations.json")
signal_sensor_map = load_json_data(SIGNAL_SENSOR_MAP_PATH)

sensor_registry_summary = pd.DataFrame(
    [
        {
            "entity": entity_name.value,
            "endpoint": default_registry.get(entity_name).endpoint,
            "record_model": default_registry.get(entity_name).record_model.__name__,
        }
        for entity_name in [
            ShmEntityName.SENSOR_TYPE,
            ShmEntityName.SENSOR,
            ShmEntityName.SENSOR_CALIBRATION,
        ]
    ]
)

sensor_type_preview = DEFAULT_SERIALIZERS[ShmEntityName.SENSOR_TYPE].to_payload(sensor_types_data[0])

sensor_dataset_summary = pd.DataFrame(
    [
        {
            "turbine": turbine,
            "sensor_rows": sum(
                len(category["cabinets"])
                for category in sensors_data[turbine].values()
                if category is not None and category.get("cabinets") is not None
            ),
            "calibration_rows": len(sensor_calibration_map.get(turbine, {})),
            "mapped_signals": len(signal_sensor_map.get(turbine, {})),
        }
        for turbine in TARGET_TURBINES
    ]
)

display(sensor_registry_summary)
display(pd.DataFrame([sensor_type_preview]))
display(sensor_dataset_summary)


,entity,endpoint,record_model
0,sensor_type,sensortype,SensorTypeRecord
1,sensor,sensor,SensorRecord
2,sensor_calibration,sensorcalibration,SensorCalibrationRecord


,name,type,type_extended,hardware_supplier,file
0,393B04,ACC,Acceleration,PCB,393b04.jpg


,turbine,sensor_rows,calibration_rows,mapped_signals
0,NRTC01,24,17,18
1,NRTC03,48,17,42


## Upload Sensor Types, Sensors, and Calibrations

This is the main dry-run upload stage.

**Execution logic**

- Build a `DryRunSensorApi` that subclasses `ShmAPI`.
- Wrap it with `ApiShmRepository` and `SensorService` so the typed retrieval path is available immediately after upload.
- Use `ShmSensorUploader` for the high-level mutation workflow.
- Upload sensor types first, then turbine-scoped sensors, then sensor calibrations.

*Outcome:* the in-memory backend contains the same resource categories that a live upload would have created.


```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
flowchart TD
    A["sensor_types_data"] --> B["ShmSensorUploader.upload_sensor_types"]
    C["sensors_data + category specs"] --> D["upload_sensors per category"]
    E["signal_sensor_map + calibration files"] --> F["upload_sensor_calibrations"]
    B --> G["DryRunSensorApi.create_sensor_type"]
    D --> H["DryRunSensorApi.create_sensor"]
    F --> I["DryRunSensorApi.create_sensor_calibration"]
    G --> J["Persisted dry-run sensor types"]
    H --> K["Persisted dry-run sensors"]
    I --> L["Persisted dry-run calibrations"]

    classDef decision fill:#FDEBD3,stroke:#C47A2C,color:#5A3412;
    classDef action fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef result fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class A,C,E decision;
    class B,D,F,G,H,I action;
    class J,K,L result;
```


In [14]:
dry_run_api = DryRunSensorApi(api_root=BASE_URL, token=TOKEN or "dry-run-token")
repository = ApiShmRepository(api=dry_run_api)
entity_service = ShmEntityService(repository=repository, registry=default_registry)
sensor_service = SensorService(entity_service=entity_service)
uploader = ShmSensorUploader(shm_api=dry_run_api)

category_specs = [
    ("accelerometers", {"name": "393B04"}),
    ("strain_gages", {"name": "6/350 CLY41-3L-10M"}),
    ("inclinometers", {"name": "NG2U"}),
    ("thermocouples", {"name": "TC0000"}),
    ("temperature", {"name": "TEMP0000"}),
    ("FBG", {"name": "FBG0000"}),
]

display(
    pd.DataFrame(
        [
            {"ping": dry_run_api.ping(), "category_count": len(category_specs), "registry_names": ", ".join(default_registry.names()[:3])}
        ]
    )
)


,ping,category_count,registry_names
0,ok,6,"derived_signal, derived_signal_calibration, de..."


In [15]:
sensor_type_results = uploader.upload_sensor_types(
    sensor_types_data=sensor_types_data,
    permission_group_ids=PERMISSION_GROUP_IDS,
    path_to_images=SENSORS_ROOT / "img",
)

sensor_results = []
for sensor_type_name, filters in category_specs:
    sensor_results.extend(
        uploader.upload_sensors(
            sensor_type_name=sensor_type_name,
            sensor_type_params=filters,
            sensors_data=sensors_data,
            permission_group_ids=PERMISSION_GROUP_IDS,
            turbines=TARGET_TURBINES,
        )
    )

sensor_calibration_results = uploader.upload_sensor_calibrations(
    signal_sensor_map_data={turbine: signal_sensor_map[turbine] for turbine in TARGET_TURBINES},
    signal_calibration_map_data=sensor_calibration_map,
    path_to_datasheets=SENSORS_ROOT / "datasheets",
    turbines=TARGET_TURBINES,
)

sensor_upload_summary = pd.DataFrame(
    [
        {
            "sensor_type_results": len(sensor_type_results),
            "sensor_results": len(sensor_results),
            "sensor_calibration_results": len(sensor_calibration_results),
            "stored_sensor_types": len(dry_run_api._sensor_types),
            "stored_sensors": len(dry_run_api._sensors),
            "stored_sensor_calibrations": len(dry_run_api._sensor_calibrations),
        }
    ]
).T.reset_index().rename(columns={"index": "metric", 0: "value"}).set_index("metric")

display(sensor_upload_summary)


,value
metric,
sensor_type_results,6
sensor_results,72
sensor_calibration_results,34
stored_sensor_types,6
stored_sensors,72
stored_sensor_calibrations,34


## Retrieve Typed Sensor Records and Verify the Round Trip

The final stage reads the dry-run rows back through the typed service and serializer stack.

**What this step does**

- List persisted sensor-domain rows through `SensorService`.
- Resolve one sensor type through `ShmQuery` to demonstrate typed filtering.
- Pull raw repository rows and deserialize them back into typed records through `DEFAULT_SERIALIZERS`.
- Assert the exact dry-run counts so the notebook acts as a regression check.

*Why this matters:* it proves that upload, repository access, registry resolution, and serializer round-tripping are all aligned.


```mermaid
%%{init: {'theme': 'base', 'themeVariables': {
  'fontSize': 'small',
  'primaryColor': '#E8F3F1',
  'primaryTextColor': '#12343B',
  'primaryBorderColor': '#2C6E6F',
  'lineColor': '#365A73',
  'tertiaryColor': '#F6F1E8',
  'clusterBkg': '#F7FAFC',
  'clusterBorder': '#B8C4CF'
}}}%%
graph TD
    A["DryRunSensorApi stored rows"] --> B["ApiShmRepository.list_records / get_record"]
    B --> C["ShmEntityService"]
    C --> D["SensorService.list_* and get_*"]
    B --> E["DEFAULT_SERIALIZERS.from_mapping"]
    D --> F["Typed sensor-domain records"]
    E --> F

    classDef api fill:#D9EEF2,stroke:#2B6F77,color:#16323A;
    classDef transform fill:#F8ECD9,stroke:#C08B3E,color:#50310E;
    classDef output fill:#EAF4E1,stroke:#5E8C61,color:#183A1D;

    class A,B,C,D api;
    class E transform;
    class F output;
```


In [16]:
sensor_type_records = sensor_service.list_sensor_types()
sensor_records = sensor_service.list_sensors()
sensor_calibration_records = sensor_service.list_sensor_calibrations()

accelerometer_type = sensor_service.get_sensor_type(
    ShmQuery(
        entity=ShmEntityName.SENSOR_TYPE,
        backend_filters={"name": "393B04"},
    )
)
raw_sensor_frame = repository.list_records(ShmEntityName.SENSOR)
first_typed_sensor = DEFAULT_SERIALIZERS[ShmEntityName.SENSOR].from_mapping(raw_sensor_frame.iloc[0].to_dict())

roundtrip_summary = pd.DataFrame(
    [
        {
            "typed_sensor_types": len(sensor_type_records),
            "typed_sensors": len(sensor_records),
            "typed_sensor_calibrations": len(sensor_calibration_records),
            "accelerometer_type_id": accelerometer_type.id if accelerometer_type is not None else None,
            "first_sensor_serial_number": first_typed_sensor.serial_number,
        }
    ]
).T.reset_index().rename(columns={"index": "metric", 0: "value"}).set_index("metric")

display(roundtrip_summary)
display(pd.DataFrame([record.model_dump(mode="json") for record in sensor_type_records[:3]]))
display(pd.DataFrame([record.model_dump(mode="json") for record in sensor_records[:3]]))
display(pd.DataFrame([record.model_dump(mode="json") for record in sensor_calibration_records[:3]]))


,value
metric,
typed_sensor_types,6
typed_sensors,72
typed_sensor_calibrations,34
accelerometer_type_id,1
first_sensor_serial_number,54652


,id,name,hardware_supplier,type,type_extended,unit,visibility,visibility_groups
0,1,393B04,PCB,ACC,Acceleration,None,usergroup,"[1, 2]"
1,2,6/350 CLY41-3L-10M,HBM,SG,Strain gauge,None,usergroup,"[1, 2]"
2,3,TC0000,supplier1,TC,Thermocouple,None,usergroup,"[1, 2]"


,id,sensor_type_id,serial_number,name,slug,cabinet,visibility,visibility_groups
0,7,1,54652,None,None,CAB11,usergroup,"[1, 2]"
1,8,1,54653,None,None,CAB11,usergroup,"[1, 2]"
2,9,1,54654,None,None,CAB12,usergroup,"[1, 2]"


,id,sensor_id,sensor_serial_number,calibration_date,value,file,status_approval
0,79,None,7,2019-02-28T00:00:00,None,Acc 54652.pdf,None
1,80,None,8,2019-02-28T00:00:00,None,Acc 54653.pdf,None
2,81,None,9,2019-02-28T00:00:00,None,Acc 54654.pdf,None


In [17]:
assert sensor_upload_summary.loc["sensor_type_results", "value"] == 6
assert sensor_upload_summary.loc["sensor_results", "value"] == 72
assert sensor_upload_summary.loc["sensor_calibration_results", "value"] == 34
assert len(sensor_type_records) == 6
assert len(sensor_records) == 72
assert len(sensor_calibration_records) == 34
assert accelerometer_type is not None and accelerometer_type.name == "393B04"
assert first_typed_sensor.sensor_type_id is not None

pd.DataFrame(
    [
        {
            "status": "ok",
            "sensor_type_count": len(sensor_type_records),
            "sensor_count": len(sensor_records),
            "sensor_calibration_count": len(sensor_calibration_records),
        }
    ]
)


,status,sensor_type_count,sensor_count,sensor_calibration_count
0,ok,6,72,34
